### 馬達論文研究
### 第四步 辨識未知資料是否為新資料
### 馬達C 11000RPM
### 使用CNN_VGG
### 先使用HDBSCAN分群再使用馬氏距離判定未知故障

In [ ]:
# --- logging bootstrap (auto-added) ---
import sys
import importlib
# In case an older bootstrap redirected outputs, restore them.
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__

import logger as _logger_mod
_logger_mod = importlib.reload(_logger_mod)
save_plot = _logger_mod.save_plot
setup_logger = _logger_mod.setup_logger
tee_std_to_file = _logger_mod.tee_std_to_file

LOG, RUN_PATHS = setup_logger('notebook', console=False)
_tee_ctx = tee_std_to_file(RUN_PATHS.log_file)
_tee_ctx.__enter__()
import atexit
atexit.register(_tee_ctx.__exit__, None, None, None)

# Auto-save matplotlib figures on plt.show()
try:
    import matplotlib.pyplot as plt
    if not getattr(plt, '_ancestor_save_plot_patched', False):
        plt._ancestor_save_plot_patched = True
        _orig_show = plt.show
        import time
        plt._ancestor_show_in_progress = False
        plt._ancestor_last_save_ts = 0.0

        def _show_and_save(*args, **kwargs):
            # Guard against backend calling show() multiple times
            if getattr(plt, '_ancestor_show_in_progress', False):
                return _orig_show(*args, **kwargs)
            now = time.monotonic()
            if now - float(getattr(plt, '_ancestor_last_save_ts', 0.0)) < 0.5:
                return _orig_show(*args, **kwargs)
            plt._ancestor_show_in_progress = True
            try:
                # Save only the current figure once
                save_plot(plt, LOG, RUN_PATHS)
            except Exception:
                pass
            try:
                return _orig_show(*args, **kwargs)
            finally:
                plt._ancestor_last_save_ts = time.monotonic()
                try:
                    plt.close(plt.gcf())
                except Exception:
                    pass
                plt._ancestor_show_in_progress = False

        plt.show = _show_and_save
    else:
        # already patched in this kernel
        pass

except Exception:
    pass
# --- end logging bootstrap ---


In [ ]:
import os
import random
import warnings
from collections import Counter, defaultdict

import hdbscan
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Model, load_model
from gpu_utils import device_scope

warnings.filterwarnings("ignore")

In [ ]:
# --------------------------- 全域參數 ---------------------------
ROOT_DIR = os.getcwd()
FEATURE_DIR = os.path.join(ROOT_DIR, 'data', 'Step-1', "myfeature")
MODEL_PATH = os.path.join(ROOT_DIR, 'data', 'Step-1', "model", "VGG16_C11000_1.keras")
RPM = "11000rpm"            # 目前只跑 C 11000 RPM，可自行改
T_CODE = "T1"              # 資料夾代號 (時間點)
LABEL_ORDER = [
    "8screws",
    "1screws",
    "2screws",
    "3screws",
    "4screws",
]  # 舊資料標籤順序

In [ ]:
# ➜ 想一次測多種設定就寫在這裡
MIN_CLUSTER_SIZES = list(range(21,25))  # 最小群集大小
CONF_LEVELS = [0.90, 0.95, 0.99]
UNKNOWN_BATCHES = [
    ["5screws"],
    ["5screws", "6screws"],
    ["5screws", "6screws", "7screws"],
    ["5screws", "6screws", "7screws", "3_14screws"],
    ["5screws", "6screws", "7screws", "3_14screws", "4_146screws"],
]

PER_SCREW_LIMIT   = 60      # 每種未知 screws 只取前 60 筆
MAX_COMBO_SAMPLE  = 450      # 混合資料後最多抽多少筆避免過大

In [ ]:
def load_and_concat(base_dir: str, t_code: str, rpm: str, screws_list):
    """讀取多個 screws 資料並 concat"""
    dfs = []
    for screws in screws_list:
        path = os.path.join(
            base_dir,
            f"{t_code}",
            rpm,
            screws,
            f"{t_code}_Group_feature_data_clean.csv",
        )
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["screws"] = screws
            dfs.append(df)
        else:
            print(f"⚠️ 缺少檔案：{path}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [ ]:
# ---------- 1. 讀取舊資料 ----------
print("\n=== 1. Loading training data ===")
train_df = load_and_concat(FEATURE_DIR, T_CODE, RPM, LABEL_ORDER)
label_map = {v: i for i, v in enumerate(LABEL_ORDER)}
train_df["label"] = train_df["screws"].map(label_map)
X = train_df.drop(["screws", "label"], axis=1).values
y = train_df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"✔️ Train : {X_train.shape}  |  Test : {X_test.shape}")


In [ ]:
# ---------- 2. CNN 特徵 ----------
cnn = load_model(MODEL_PATH, compile=False)
feat_model = Model(inputs=cnn.input, outputs=Flatten()(cnn.get_layer("max_pooling1d").output))
X_train_f = feat_model.predict(X_train_s.reshape(-1, X_train_s.shape[1], 1), verbose=0)
X_test_f  = feat_model.predict(X_test_s.reshape(-1, X_test_s.shape[1], 1),  verbose=0)

In [ ]:
# ---------- 3. 以不同 min_cluster_size 建 HDBSCAN & 清離群 ----------
def fit_hdbscan_inliers(features, labels, mcs):
    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, cluster_selection_method="eom").fit(features)
    mask = cl.labels_ != -1
    return features[mask], labels[mask]

In [ ]:
# ---------- 4. Mahalanobis 工具函式 ----------
def mahalanobis_stats(X, conf):
    mu  = X.mean(axis=0)
    cov = np.cov(X, rowvar=False)
    inv = np.linalg.pinv(cov + np.eye(cov.shape[0]) * 1e-6)
    d   = np.sqrt(((X - mu) @ inv * (X - mu)).sum(axis=1))
    thr = np.percentile(d, conf * 100)
    return mu, inv, thr


In [ ]:
# ---------- 5‑A. Test‑set validation ----------
def validate_test_set(features, true_labels, mcs, mu, inv, thr):
    print("\n--- Test‑set validation")
    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, cluster_selection_method="leaf").fit(features)
    lbl = cl.labels_
    for cid in sorted(set(lbl)):
        idx = np.where(lbl == cid)[0]
        names = [LABEL_ORDER[l] for l in true_labels[idx]]
        cnts = Counter(names)
        tag  = "outliers" if cid == -1 else f"C{cid}"
        info = "，".join(f"{k} {v}筆" for k, v in cnts.items())
        print(f" {tag:<8}: {info}")
    for cid in sorted(set(lbl) - {-1}):
        idx  = np.where(lbl == cid)[0]
        ctr  = features[idx].mean(axis=0, keepdims=True)
        dist = np.sqrt(((ctr - mu) @ inv * (ctr - mu)).sum())
        stat = "New" if dist > thr else "Known"
        print(f"  * C{cid:<2} | size {len(idx):3d} | dist {dist:6.2f} | {stat}")

In [ ]:
# ---------- 5-B. Unknown data pipeline ----------
def evaluate_unknown_batch(batch, mcs, mu, inv, thr):
    unknown_df = load_and_concat(FEATURE_DIR, T_CODE, RPM, batch)
    if unknown_df.empty:
        print("❌ 無資料，跳過")
        return
    # 每種 screws 取前 60 筆
    unknown_df = unknown_df.groupby("screws", group_keys=False).head(PER_SCREW_LIMIT)

    # 混入舊 Test set（標示 old_test）方便觀察
    old_df = pd.DataFrame(X_test, columns=train_df.columns.drop(["screws", "label"]))
    old_df["screws"] = "old_test"
    combo = pd.concat([old_df, unknown_df], ignore_index=True)
    combo = combo.sample(min(len(combo), MAX_COMBO_SAMPLE), random_state=42)

    X_new_s = scaler.transform(combo.drop("screws", axis=1).values)
    X_new_f = feat_model.predict(X_new_s.reshape(-1, X_new_s.shape[1], 1), verbose=0)

    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, cluster_selection_method="leaf").fit(X_new_f)
    lbl = cl.labels_
    cnt = Counter(lbl)

    print(f"\n--- Unknown {batch} | mcs {mcs}")
    for k, v in cnt.items():
        tag = "outliers" if k == -1 else f"C{k}"
        print(f"  {tag:<8}: {v} pts")
    for cid in sorted(set(lbl) - {-1}):
        idx  = np.where(lbl == cid)[0]
        ctr  = X_new_f[idx].mean(axis=0, keepdims=True)
        dist = np.sqrt(((ctr - mu) @ inv * (ctr - mu)).sum())
        stat = "New Fault" if dist > thr else "Known"
        print(f"  C{cid:<2} | size {len(idx):3d} | dist {dist:6.2f} | {stat}")

In [ ]:
# -------------------- 主程式迴圈 --------------------
for mcs in MIN_CLUSTER_SIZES:
    X_filt, y_filt = fit_hdbscan_inliers(X_train_f, y_train, mcs)
    for conf in CONF_LEVELS:
        MU, INV, THR = mahalanobis_stats(X_filt, conf)
        print("\n==============================")
        print(f"▶ mcs = {mcs} | conf = {conf} | thr = {THR:.2f}")
        print("==============================")
        validate_test_set(X_test_f, y_test, mcs, MU, INV, THR)
        for batch in UNKNOWN_BATCHES:
            evaluate_unknown_batch(batch, mcs, MU, INV, THR)